Práctica 05: Simulación de Acumulación de Ruido (ASE + NLI) en Redes Ópticas
Basado en los principios de GNPy y el Modelo GN


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def simular_red_gn(num_spans, p_ch_dbm, span_loss_db, nf_db):
    """
    Calcula la GSNR acumulada considerando ruido ASE de amplificadores
    y ruido NLI generado por la fibra (Efecto Kerr).
    """
    # 1. Convertir dBm/dB a lineal
    p_ch_lin = 10**(p_ch_dbm / 10.0) / 1000 # Watts
    nf_lin = 10**(nf_db / 10.0)

    # 2. Constantes para Ruido ASE (Amplificadores)
    # P_ase = h * nu * B * (G-1) * NF
    # Simplificado para Banda C (1550nm):
    p_ase_lin = 10**(-58/10) * nf_lin * (10**(span_loss_db/10)) / 1000

    # 3. Constantes para Ruido NLI (No linealidad de la fibra)
    # En el modelo GN, NLI es proporcional a P_ch^3
    eta_nli = 1e-4 # Coeficiente de eficiencia NLI típico para G.652
    p_nli_lin = eta_nli * (p_ch_lin**3)

    # 4. Acumulación a lo largo de los spans
    gsnr_acumulada = []
    ruido_total_lin = 0

    for i in range(1, num_spans + 1):
        # El ruido se suma linealmente en cada salto
        ruido_total_lin += (p_ase_lin + p_nli_lin)
        gsnr_db = 10 * np.log10(p_ch_lin / ruido_total_lin)
        gsnr_acumulada.append(gsnr_db)

    return gsnr_acumulada

# --- Escenario de Laboratorio ---
spans = np.arange(1, 21, 1) # Simulación de 1 a 20 saltos de 80km
potencia_optima = 0.0      # Potencia sugerida en dBm
atenuacion_tramo = 17.6     # 80km * 0.22 dB/km
figura_ruido = 6.0         # EDFA NF típico

# Ejecutar Simulación
resultados_gsnr = simular_red_gn(20, potencia_optima, atenuacion_tramo, figura_ruido)

# --- Visualización de Resultados ---
plt.figure(figsize=(10, 6))
plt.plot(spans, resultados_gsnr, 'D-', color='darkblue', label='GSNR Total (ASE + NLI)')
plt.axhline(y=18, color='red', linestyle='--', label='Umbral 16-QAM (Crítico)')
plt.axhline(y=12, color='orange', linestyle='--', label='Umbral QPSK (Respaldo)')

plt.title("Calidad de Transmisión (QoT) vs Número de Tramos (Spans)")
plt.xlabel("Número de Saltos (Amplificadores EDFA)")
plt.ylabel("GSNR (dB)")
plt.xticks(spans)
plt.legend()
plt.grid(True, which='both', linestyle=':', alpha=0.5)
plt.fill_between(spans, 18, 25, color='green', alpha=0.1, label='Zona Operativa 16-QAM')
plt.show()

print(f"GSNR tras 10 saltos (800km): {resultados_gsnr[1]:.2f} dB")

## Análisis del Código, Ejecución y Resultados

### Análisis del Código (`simular_red_gn`)

El código implementa una función `simular_red_gn` que calcula la GSNR (Generalized Signal-to-Noise Ratio) acumulada en una red óptica, considerando dos fuentes principales de ruido: el ruido ASE (Amplified Spontaneous Emission) generado por los amplificadores ópticos (EDFA) y el ruido NLI (Non-Linear Interference) resultante de los efectos no lineales de la fibra óptica (efecto Kerr).

1.  **Conversión de Unidades**: Inicialmente, las potencias y figuras de ruido en dBm/dB se convierten a escalas lineales (Watts y lineal, respectivamente) para facilitar los cálculos de acumulación de ruido.
2.  **Cálculo de Ruido ASE**: Se utiliza una fórmula simplificada para el ruido ASE en la Banda C (1550nm), que depende de la figura de ruido del amplificador y la ganancia (que en un sistema compensado es igual a la pérdida del tramo).
3.  **Cálculo de Ruido NLI**: El modelo GN (Gaussian Noise) se aplica para estimar el ruido NLI, que es proporcional al cubo de la potencia de canal (`p_ch_lin^3`), utilizando un coeficiente de eficiencia NLI (`eta_nli`) típico para fibra G.652.
4.  **Acumulación de Ruido y GSNR**: El ruido total (ASE + NLI) se acumula linealmente en cada tramo. La GSNR se calcula en dB como `10 * log10(P_ch / Ruido_Total_Lineal)`. La función devuelve una lista con los valores de GSNR para cada tramo simulado.

### Análisis de la Ejecución y Configuración

El script simula una red con las siguientes características:

*   **Número de Tramos (Spans)**: De 1 a 20 tramos.
*   **Potencia Óptima por Canal (`potencia_optima`)**: 0.0 dBm. Esta es una potencia de lanzamiento común en redes ópticas.
*   **Atenuación por Tramo (`atenuacion_tramo`)**: 17.6 dB, lo que sugiere tramos de fibra de aproximadamente 80 km (considerando una atenuación de 0.22 dB/km).
*   **Figura de Ruido del Amplificador (`figura_ruido`)**: 6.0 dB, un valor típico para amplificadores EDFA.

La simulación calcula la GSNR para estos 20 tramos y luego visualiza los resultados.

### Análisis de los Resultados Generados

*   **Gráfica de GSNR vs. Número de Tramos**: La gráfica muestra claramente que la GSNR disminuye a medida que aumenta el número de tramos. Esto es un comportamiento esperado, ya que tanto el ruido ASE como el NLI se acumulan a lo largo de la red, degradando la calidad de la señal.
*   **Umbrales de Modulación**: Se han añadido líneas horizontales para los umbrales de GSNR de dos formatos de modulación: 18 dB para 16-QAM (Crítico) y 12 dB para QPSK (Respaldo). Una zona sombreada en verde entre 18 dB y 25 dB indica la 'Zona Operativa 16-QAM'.
*   **Valor Específico**: El script imprime la GSNR tras 10 tramos (800 km) como **31.39 dB**.

---

## Resultados, Conclusiones y Recomendaciones

### Resultados

La simulación demuestra que, con los parámetros de red definidos (potencia de 0 dBm, atenuación de 17.6 dB/tramo, figura de ruido de 6 dB), la GSNR se mantiene en niveles elevados incluso después de 20 tramos. Específicamente, a los 10 tramos (equivalentes a 800 km), la GSNR es de 31.39 dB. La GSNR decrece de forma consistente a lo largo de la distancia, pasando de un valor superior a 34 dB en el primer tramo a aproximadamente 22 dB en el tramo 20. En todo el rango simulado (hasta 20 tramos), la GSNR se mantiene muy por encima del umbral crítico de 18 dB para 16-QAM y del umbral de 12 dB para QPSK.

### Conclusiones

1.  **Viabilidad de Transmisión**: La red simulada es robusta y capaz de soportar la transmisión con modulación 16-QAM (y, por supuesto, QPSK) en distancias de hasta al menos 20 tramos (1600 km), manteniendo un margen de GSNR saludable por encima de los umbrales críticos.
2.  **Dominancia del Ruido**: El modelo GN utilizado predice la acumulación de ruido ASE y NLI, mostrando cómo esta acumulación degrada la GSNR con la distancia. Para esta configuración específica, la calidad de la transmisión es excelente.
3.  **Margen de Diseño**: La GSNR obtenida en 20 tramos (aproximadamente 22 dB) ofrece un margen considerable sobre el umbral de 16-QAM, lo que podría permitir tolerar ciertas degradaciones o la utilización de formatos de modulación ligeramente superiores (si los umbrales de GSNR lo permiten).

### Recomendaciones

1.  **Optimización de Parámetros**: Investigar el impacto de variar la potencia de lanzamiento (`potencia_optima`) para identificar el punto óptimo real donde se minimiza la degradación total (equilibrio entre ASE y NLI). Esto es crucial para un diseño eficiente.
2.  **Exploración de Distancias Mayores o Formatos de Modulación Superiores**: Dada la GSNR obtenida, se podría simular la red para un mayor número de tramos o evaluar la viabilidad de formatos de modulación de orden superior (ej. 64-QAM) si se conocieran sus umbrales de GSNR.
3.  **Análisis de Contribución de Ruido**: Modificar el código para graficar las contribuciones individuales de ruido ASE y NLI. Esto ayudaría a entender qué tipo de ruido es dominante en diferentes partes de la red y cómo afecta la potencia de lanzamiento.
4.  **Comparación con Otros Modelos**: Si fuera posible, comparar estos resultados con modelos más complejos o herramientas como GNPy para validar la precisión del modelo GN simplificado.